# Project M3S - Swarm Commander Training Notebook
## Fine-tuning `Qwen/Qwen2.5-Coder-0.5B-Instruct` for High-Speed Deterministic Swarm DSL
- Framework: Unsloth / HuggingFace TRL (Fast QLoRA)
- Output: 4-bit Quantized GGUF (`m3s_commander_q4.gguf` ~300MB)
- Runtime: Google Colab Free T4 GPU (~15 minutes)

In [ ]:
# 1. Install Unsloth & Dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
# 2. Load Qwen2.5-Coder-0.5B-Instruct
from unsloth import FastLanguageModel
import torch

max_seq_length = 512
model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

# 3. Add LoRA Adapters specifically targeting Attention & MLP layers
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Load & Format Dataset (m3s_swarm_dsl_train.jsonl)
from datasets import load_dataset

# Upload your m3s_swarm_dsl_train.jsonl to Colab
dataset = load_dataset("json", data_files="m3s_swarm_dsl_train.jsonl", split="train")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 5. Train Model (SFTTrainer)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        warmup_steps = 20,
        max_steps = 250, # Fast convergence on structured DSL
        learning_rate = 3e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
# 6. Export directly to 4-bit Quantized GGUF for ThinkCentre M920s
# Output file: m3s_commander_q4.gguf (~300MB)
model.save_pretrained_gguf("m3s_commander_q4", tokenizer, quantization_method = "q4_k_m")
print("Training Complete! Download m3s_commander_q4/m3s_commander_q4-unsloth.Q4_K_M.gguf and load into Ollama.")